### Adding GPU

In [1]:
##import os
##os.environ["CUDA_VISIBLE_DEVICES"]="XX"

### Importing the necessary libraries

In [ ]:
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import json
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")

### Loading Biobert and its Tokenizer

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3  # 3 labels yes,no,maybe
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Loading BioBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

### Loading the datasets for training

In [3]:
def load_from_json(file_path):
   
    with open(file_path, "r") as f:
        data_dict = json.load(f)
    return Dataset.from_dict(data_dict)

In [4]:
# Load MRSTY datasets
train_dataset_mrsty = load_from_json("final_biobert/train_dataset_mrsty.json")
dev_dataset_mrsty = load_from_json("final_biobert/dev_dataset_mrsty.json")
test_dataset_mrsty = load_from_json("final_biobert/test_dataset_mrsty.json")

# Load MRREL datasets
train_dataset_mrrel = load_from_json("final_biobert/train_dataset_mrrel.json")
dev_dataset_mrrel = load_from_json("final_biobert/dev_dataset_mrrel.json")
test_dataset_mrrel = load_from_json("final_biobert/test_dataset_mrrel.json")

print("Datasets loaded successfully")

Datasets loaded successfully


### Function for computing metrics

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

### Baseline Biobert without UMLS data

#### Loading and preprocessing the data without UMLS

In [8]:
from datasets import Dataset

label_mapping = {"yes": 0, "no": 1, "maybe": 2}

def load_and_process_data(filepath, tokenizer, label_mapping, max_length=512):
    """
    Load and process the preprocessed dataset for BioBERT.
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    processed_data = {
        "input_ids": [],
        "attention_mask": [],
        "label": []
    }
    
    for pmid, entry in tqdm(data.items(), desc="Processing data"):
        question = entry.get("question", "")
        context = entry.get("context", "")
        label = entry.get("label", None)
        
        if label is None or question == "" or context == "":
            # Skip entries with missing fields
            continue
        
        # Tokenize the question and context
        inputs = tokenizer(
            question + " " + context,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="np"
        )
        
        processed_data["input_ids"].append(inputs["input_ids"][0])
        processed_data["attention_mask"].append(inputs["attention_mask"][0])
        processed_data["label"].append(label_mapping[label])
    
    return Dataset.from_dict(processed_data)


In [9]:
train_dataset_bio = load_and_process_data("preprocessed/processed_train_set.json", tokenizer, label_mapping)
dev_dataset_bio = load_and_process_data("preprocessed/processed_dev_set.json", tokenizer, label_mapping)
test_dataset_bio = load_and_process_data("preprocessed/processed_test_set.json", tokenizer, label_mapping)


Processing data: 100%|███████████████████████| 100/100 [00:00<00:00, 900.22it/s]


In [10]:
training_args_bio = TrainingArguments(
    output_dir="./results_baseline",  # Output directory for model checkpoints
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs_baseline",  # Directory for logging
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5  
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
trainer_bio = Trainer(
    model=model,
    args=training_args_bio,
    train_dataset=train_dataset_bio,
    eval_dataset=dev_dataset_bio,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [12]:
# Train and evaluate
print("Starting training for Baseline...")
trainer_bio.train()
print("Evaluating Baseline on test set...")
results_bio = trainer_bio.evaluate(test_dataset_bio)
print("Baseline Test Results:", results_bio)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training for Baseline...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.957200,1.025416,0.510000,0.344503,0.260100,0.510000
2,0.871700,0.974729,0.570000,0.523204,0.489394,0.570000
3,0.686100,0.984983,0.590000,0.541728,0.510204,0.590000
4,0.497400,1.072507,0.570000,0.523827,0.493673,0.570000
5,0.449700,1.143543,0.570000,0.527009,0.519886,0.570000
6,0.318100,1.224835,0.580000,0.528569,0.486519,0.580000
7,0.332600,1.232224,0.600000,0.549214,0.512633,0.600000
8,0.273100,1.345609,0.610000,0.558408,0.522885,0.610000
9,0.206600,1.354341,0.600000,0.549214,0.512633,0.600000
10,0.179200,1.353356,0.600000,0.552000,0.520000,0.600000


Evaluating Baseline on test set...


Baseline Test Results: {'eval_loss': 1.2321748733520508, 'eval_accuracy': 0.64, 'eval_f1': 0.6148815482148815, 'eval_precision': 0.593262987012987, 'eval_recall': 0.64, 'eval_runtime': 5.35, 'eval_samples_per_second': 18.692, 'eval_steps_per_second': 2.43, 'epoch': 10.0}


In [13]:
# Save trained model and tokenizer
trainer_bio.save_model("./model_baseline")
tokenizer.save_pretrained("./tokenizer_baseline")

('./tokenizer_baseline/tokenizer_config.json',
 './tokenizer_baseline/special_tokens_map.json',
 './tokenizer_baseline/vocab.txt',
 './tokenizer_baseline/added_tokens.json',
 './tokenizer_baseline/tokenizer.json')

### Training with UMLS info

#### MRSTY and MRCONSO

In [6]:
model_mrsty_carry = AutoModelForSequenceClassification.from_pretrained("./model_baseline")
tokenizer_mrsty_carry = AutoTokenizer.from_pretrained("./tokenizer_baseline")

In [7]:
training_args_mrsty_carry = TrainingArguments(
    output_dir="./results_mrsty_carry",  # Output directory for model checkpoints
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs_mrsty_carry",  # Directory for logging
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3  
)

In [8]:
trainer_mrsty_carry = Trainer(
    model=model_mrsty_carry,
    args=training_args_mrsty_carry,
    train_dataset=train_dataset_mrsty ,
    eval_dataset=dev_dataset_mrsty ,
    tokenizer=tokenizer_mrsty_carry,
    compute_metrics=compute_metrics
)

In [9]:
print("Starting training...")
trainer_mrsty_carry.train()
print("Evaluating on the test set...")
results_mrsty_carry = trainer_mrsty_carry.evaluate(test_dataset_mrsty)
print("Test Results:", results_mrsty_carry)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.396500,0.197817,0.925000,0.908655,0.930957,0.925000
2,0.148000,0.086984,0.973750,0.972444,0.974041,0.973750
3,0.216100,0.050912,0.986250,0.985833,0.986483,0.986250
4,0.020000,0.018161,0.995000,0.994951,0.995044,0.995000
5,0.020900,0.012381,0.996250,0.996222,0.996271,0.996250


Evaluating on the test set...


Test Results: {'eval_loss': 2.2815699577331543, 'eval_accuracy': 0.57, 'eval_f1': 0.574931367209864, 'eval_precision': 0.5802222222222222, 'eval_recall': 0.57, 'eval_runtime': 5.345, 'eval_samples_per_second': 18.709, 'eval_steps_per_second': 2.432, 'epoch': 5.0}


In [11]:
# Save trained model and tokenizer
trainer_mrsty_carry.save_model("./model_mrsty_carry")
tokenizer_mrsty_carry.save_pretrained("./tokenizer_mrsty_carry")

('./tokenizer_mrsty_carry/tokenizer_config.json',
 './tokenizer_mrsty_carry/special_tokens_map.json',
 './tokenizer_mrsty_carry/vocab.txt',
 './tokenizer_mrsty_carry/added_tokens.json',
 './tokenizer_mrsty_carry/tokenizer.json')

#### Without Carrying information

In [12]:
model_mrsty = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3  # 3 labels yes,no,maybe
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# Loading BioBERT tokenizer
tokenizer_mrsty = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

In [14]:
training_args_mrsty = TrainingArguments(
    output_dir="./results_mrsty",  # Output directory for model checkpoints
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs_mrsty",  # Directory for logging
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5  
)

In [15]:
trainer_mrsty = Trainer(
    model=model_mrsty,
    args=training_args_mrsty,
    train_dataset=train_dataset_mrsty ,
    eval_dataset=dev_dataset_mrsty ,
    tokenizer=tokenizer_mrsty,
    compute_metrics=compute_metrics
)

In [16]:
print("Starting training...")
trainer_mrsty.train()
print("Evaluating on the test set...")
results_mrsty = trainer_mrsty.evaluate(test_dataset_mrsty)
print("Test Results:", results_mrsty)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.959100,0.923781,0.557500,0.399109,0.310806,0.557500
2,0.945300,0.919911,0.557500,0.399109,0.310806,0.557500
3,0.831100,0.835489,0.652500,0.599902,0.591998,0.652500
4,0.793500,0.725797,0.710000,0.669628,0.633988,0.710000
5,0.651900,0.605388,0.790000,0.745661,0.706415,0.790000
6,0.581900,0.494558,0.836250,0.790378,0.756602,0.836250
7,0.465700,0.394900,0.868750,0.821054,0.780663,0.868750
8,0.432400,0.354380,0.873750,0.825859,0.784993,0.873750
9,0.352000,0.327031,0.876250,0.828008,0.786296,0.876250
10,0.312400,0.319326,0.877500,0.829088,0.786880,0.877500


Evaluating on the test set...


Test Results: {'eval_loss': 1.0766452550888062, 'eval_accuracy': 0.59, 'eval_f1': 0.5679100529100529, 'eval_precision': 0.5521437173825773, 'eval_recall': 0.59, 'eval_runtime': 5.3705, 'eval_samples_per_second': 18.62, 'eval_steps_per_second': 2.421, 'epoch': 10.0}


In [18]:
# Save trained model and tokenizer
trainer_mrsty.save_model("./model_mrsty")
tokenizer_mrsty.save_pretrained("./tokenizer_mrsty")

('./tokenizer_mrsty/tokenizer_config.json',
 './tokenizer_mrsty/special_tokens_map.json',
 './tokenizer_mrsty/vocab.txt',
 './tokenizer_mrsty/added_tokens.json',
 './tokenizer_mrsty/tokenizer.json')

#### MRREL

In [19]:
model_mrrel_carry = AutoModelForSequenceClassification.from_pretrained("./model_mrsty_carry")
tokenizer_mrrel_carry = AutoTokenizer.from_pretrained("./tokenizer_mrsty_carry")

In [20]:
training_args_mrrel_carry = TrainingArguments(
    output_dir="./results_mrrel_carry",  # Output directory for model checkpoints
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs_mrrel_carry",  # Directory for logging
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3  
)

In [21]:
trainer_mrrel_carry = Trainer(
    model=model_mrrel_carry,
    args=training_args_mrrel_carry,
    train_dataset=train_dataset_mrrel,
    eval_dataset=dev_dataset_mrrel,
    tokenizer=tokenizer_mrrel_carry,
    compute_metrics=compute_metrics
)

In [22]:
print("Starting training...")
trainer_mrrel_carry.train()
print("Evaluating on the test set...")
results_mrrel_carry = trainer_mrrel_carry.evaluate(test_dataset_mrrel )
print("Test Results:", results_mrrel_carry)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.040400,2.530191,0.550000,0.535686,0.546925,0.550000
2,0.063000,2.504157,0.590000,0.569285,0.556601,0.590000
3,0.001700,2.716174,0.520000,0.528445,0.547463,0.520000
4,0.001100,2.758327,0.540000,0.540244,0.546603,0.540000
5,0.005400,2.797580,0.560000,0.558542,0.564667,0.560000


Evaluating on the test set...


Test Results: {'eval_loss': 2.5826375484466553, 'eval_accuracy': 0.59, 'eval_f1': 0.5876941832114246, 'eval_precision': 0.5935522894290559, 'eval_recall': 0.59, 'eval_runtime': 5.3563, 'eval_samples_per_second': 18.669, 'eval_steps_per_second': 2.427, 'epoch': 5.0}


In [23]:
trainer_mrrel_carry.save_model("./model_mrrel_carry")
tokenizer_mrrel_carry.save_pretrained("./tokenizer_mrrel_carry")

('./tokenizer_mrrel_carry/tokenizer_config.json',
 './tokenizer_mrrel_carry/special_tokens_map.json',
 './tokenizer_mrrel_carry/vocab.txt',
 './tokenizer_mrrel_carry/added_tokens.json',
 './tokenizer_mrrel_carry/tokenizer.json')

#### Without carrying information

In [24]:
model_mrrel = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3  # 3 labels yes,no,maybe
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
# Loading BioBERT tokenizer
tokenizer_mrrel = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

In [26]:
training_args_mrrel = TrainingArguments(
    output_dir="./results_mrrel",  # Output directory for model checkpoints
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs_mrrel_carry",  # Directory for logging
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5 
)

In [27]:
trainer_mrrel = Trainer(
    model=model_mrrel,
    args=training_args_mrrel,
    train_dataset=train_dataset_mrrel,
    eval_dataset=dev_dataset_mrrel,
    tokenizer=tokenizer_mrrel,
    compute_metrics=compute_metrics
)

In [28]:
print("Starting training...")
trainer_mrrel.train()
print("Evaluating on the test set...")
results_mrrel = trainer_mrrel.evaluate(test_dataset_mrrel )
print("Test Results:", results_mrrel)

Starting training...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.954100,1.028847,0.510000,0.344503,0.260100,0.510000
2,0.932000,1.030212,0.510000,0.344503,0.260100,0.510000
3,0.763600,1.001377,0.540000,0.488734,0.446551,0.540000
4,0.615500,0.985247,0.600000,0.549630,0.516735,0.600000
5,0.520000,0.968020,0.610000,0.559630,0.526735,0.610000
6,0.436200,0.979198,0.650000,0.589284,0.538973,0.650000
7,0.350200,1.037151,0.640000,0.585314,0.540029,0.640000
8,0.261400,1.184404,0.610000,0.559630,0.526735,0.610000
9,0.248300,1.187865,0.610000,0.558408,0.522885,0.610000
10,0.202100,1.214525,0.600000,0.549630,0.516735,0.600000


Evaluating on the test set...


Test Results: {'eval_loss': 0.9995507001876831, 'eval_accuracy': 0.63, 'eval_f1': 0.5946753246753247, 'eval_precision': 0.5782467217095677, 'eval_recall': 0.63, 'eval_runtime': 5.3637, 'eval_samples_per_second': 18.644, 'eval_steps_per_second': 2.424, 'epoch': 10.0}


In [29]:
trainer_mrrel.save_model("./model_mrrel")
tokenizer_mrrel.save_pretrained("./tokenizer_mrrel")

('./tokenizer_mrrel/tokenizer_config.json',
 './tokenizer_mrrel/special_tokens_map.json',
 './tokenizer_mrrel/vocab.txt',
 './tokenizer_mrrel/added_tokens.json',
 './tokenizer_mrrel/tokenizer.json')

### Fine-Tuning


#### Using the same tokenizer from carry but different model

In [6]:
tokenizer_mrrel_carry = AutoTokenizer.from_pretrained("./tokenizer_mrrel_carry")

In [7]:
model_mrrel_fine = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1",
    num_labels=3  # 3 labels yes,no,maybe
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
training_args_mrrel_fine_1 = TrainingArguments(
    output_dir="./results_mrrel_fine_1",  # Output directory for model checkpoints
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model checkpoint after every epoch
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs_mrrel_fine_1",  # Directory for logging
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5 
)

In [9]:
trainer_mrrel_fine_1 = Trainer(
    model=model_mrrel_fine,
    args=training_args_mrrel_fine_1,
    train_dataset=train_dataset_mrrel,
    eval_dataset=dev_dataset_mrrel,
    tokenizer=tokenizer_mrrel_carry,
    compute_metrics=compute_metrics
)

In [10]:
print("Starting training...")
trainer_mrrel_fine_1.train()
print("Evaluating on the test set...")
results_mrrel_fine_1 = trainer_mrrel_fine_1.evaluate(test_dataset_mrrel)
print("Test Results:", results_mrrel_fine_1)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.970800,1.026111,0.510000,0.344503,0.260100,0.510000
2,0.847700,0.987638,0.540000,0.479813,0.437381,0.540000
3,0.704200,0.957891,0.570000,0.511040,0.468205,0.570000
4,0.545000,1.040250,0.540000,0.494388,0.460242,0.540000
5,0.480900,1.110423,0.560000,0.513396,0.479771,0.560000
6,0.360900,1.087665,0.600000,0.543856,0.498950,0.600000
7,0.288000,1.159855,0.540000,0.494689,0.457040,0.540000
8,0.261300,1.213653,0.560000,0.511980,0.471831,0.560000
9,0.235200,1.247053,0.540000,0.499695,0.469392,0.540000
10,0.174100,1.237426,0.530000,0.503461,0.508030,0.530000


Evaluating on the test set...


Test Results: {'eval_loss': 0.9691064953804016, 'eval_accuracy': 0.64, 'eval_f1': 0.5964503816793894, 'eval_precision': 0.5959096109839817, 'eval_recall': 0.64, 'eval_runtime': 5.3684, 'eval_samples_per_second': 18.627, 'eval_steps_per_second': 2.422, 'epoch': 10.0}


In [11]:
trainer_mrrel_fine_1.save_model("./model_mrrel_fine_1")

#### Using Dropouts to see if any improvements

In [6]:
model_mrsty = AutoModelForSequenceClassification.from_pretrained("./model_mrsty")
model_mrrel = AutoModelForSequenceClassification.from_pretrained("./model_mrrel")

In [7]:
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn

class BioBERTWithDropout(nn.Module):
    def __init__(self, model, num_labels):
        super().__init__()
        self.bert = model.bert
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits
        )


In [8]:
# Add Dropout to MRSTY model
fine_model_mrsty = BioBERTWithDropout(model_mrsty, num_labels=3)

# Add Dropout to MRREL model
fine_model_mrrel = BioBERTWithDropout(model_mrrel, num_labels=3)

In [12]:
training_args_mrsty_2 = TrainingArguments(
    output_dir="./results_mrsty_2",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,  # Adjusted learning rate
    warmup_steps=500,  # Add warm-up
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.1,  # Increased weight decay
    logging_dir="./logs_mrsty_2",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5,
    report_to="none"  # Disable reporting to WandB or other services
)


In [9]:
training_args_mrrel_2 = TrainingArguments(
    output_dir="./results_mrrel_2",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,  # Adjusted learning rate
    warmup_steps=500,  # Add warm-up
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.1,  # Increased weight decay
    logging_dir="./logs_mrrel_2",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5,
    report_to="none"  # Disable reporting to WandB or other services
)


In [10]:
tokenizer_mrsty = AutoTokenizer.from_pretrained("./tokenizer_mrsty")
tokenizer_mrrel = AutoTokenizer.from_pretrained("./tokenizer_mrrel")

In [13]:
# Initialize the trainer with the custom loss
trainer_mrsty_2 = Trainer(
    model=fine_model_mrsty,
    args=training_args_mrsty_2,
    train_dataset=train_dataset_mrsty,
    eval_dataset=dev_dataset_mrsty,
    tokenizer=tokenizer_mrsty,
    compute_metrics=compute_metrics
)

trainer_mrrel_2 = Trainer(
    model=fine_model_mrrel,
    args=training_args_mrrel_2,
    train_dataset=train_dataset_mrrel,
    eval_dataset=dev_dataset_mrrel,
    tokenizer=tokenizer_mrrel,
    compute_metrics=compute_metrics
)



In [13]:
# Train and evaluate
print("Starting training for MRSTY...")
trainer_mrsty_2.train()
print("Evaluating MRSTY on test set...")
results_mrsty_2 = trainer_mrsty_2.evaluate(test_dataset_mrsty)
print("MRSTY Test Results:", results_mrsty_2)


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training for MRSTY...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.770000,0.648233,0.850000,0.803505,0.766884,0.850000
2,0.538200,0.391769,0.861250,0.819849,0.875382,0.861250
3,0.362400,0.226751,0.897500,0.864107,0.910111,0.897500
4,0.461100,0.239817,0.903750,0.892806,0.914400,0.903750
5,0.521700,0.199118,0.947500,0.942232,0.949080,0.947500
6,0.210500,0.095491,0.981250,0.980564,0.981617,0.981250
7,0.068800,0.025608,0.995000,0.994947,0.995030,0.995000
8,0.005300,0.009395,0.998750,0.998747,0.998753,0.998750
9,0.002700,0.008609,0.998750,0.998747,0.998755,0.998750
10,0.001000,0.004111,0.998750,0.998747,0.998753,0.998750


Evaluating MRSTY on test set...


MRSTY Test Results: {'eval_loss': 2.6451756954193115, 'eval_accuracy': 0.57, 'eval_f1': 0.5624846834581348, 'eval_precision': 0.5568620689655173, 'eval_recall': 0.57, 'eval_runtime': 5.3697, 'eval_samples_per_second': 18.623, 'eval_steps_per_second': 2.421, 'epoch': 10.0}


In [14]:
trainer_mrsty_2.save_model("./model_mrsty_2")
tokenizer_mrsty.save_pretrained("./tokenizer_mrsty_2")

('./tokenizer_mrsty_2/tokenizer_config.json',
 './tokenizer_mrsty_2/special_tokens_map.json',
 './tokenizer_mrsty_2/vocab.txt',
 './tokenizer_mrsty_2/added_tokens.json',
 './tokenizer_mrsty_2/tokenizer.json')

In [14]:
print("Starting training for MRREL...")
trainer_mrrel_2.train()
print("Evaluating MRREL on test set...")
results_mrrel_2 = trainer_mrrel_2.evaluate(test_dataset_mrrel)
print("MRREL Test Results:", results_mrrel_2)


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting training for MRREL...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.604100,0.978810,0.600000,0.550881,0.520800,0.600000
2,0.361300,1.158806,0.590000,0.537218,0.494138,0.590000
3,0.277600,1.609913,0.560000,0.513844,0.476694,0.560000
4,0.072400,1.903009,0.530000,0.515140,0.531850,0.530000
5,0.047400,2.292379,0.600000,0.568200,0.584694,0.600000
6,0.130900,2.532292,0.540000,0.513829,0.500571,0.540000
7,0.013600,2.836065,0.580000,0.553380,0.557669,0.580000
8,0.090300,2.885225,0.550000,0.521136,0.510545,0.550000
9,0.001000,3.110266,0.560000,0.516020,0.479345,0.560000
10,0.071900,3.133167,0.550000,0.514545,0.513571,0.550000


Evaluating MRREL on test set...


MRREL Test Results: {'eval_loss': 2.231934070587158, 'eval_accuracy': 0.59, 'eval_f1': 0.5727426160337552, 'eval_precision': 0.5572023809523811, 'eval_recall': 0.59, 'eval_runtime': 5.3609, 'eval_samples_per_second': 18.654, 'eval_steps_per_second': 2.425, 'epoch': 10.0}


In [15]:
trainer_mrrel_2.save_model("./model_mrrel_2")
tokenizer_mrrel.save_pretrained("./tokenizer_mrrel_2")

('./tokenizer_mrrel_2/tokenizer_config.json',
 './tokenizer_mrrel_2/special_tokens_map.json',
 './tokenizer_mrrel_2/vocab.txt',
 './tokenizer_mrrel_2/added_tokens.json',
 './tokenizer_mrrel_2/tokenizer.json')